In [25]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from datetime import datetime, timedelta

In [26]:
# Set seed for reproducibility so your numbers stay consistent
np.random.seed(42)

In [27]:
#Configure & Parameters

NUM_ORDERS = 5000
START_DATE = datetime(2026,6,1)


#Real Barcelona neighborhoods to give it a hyperlocal feel
NEIGHBORHOODS = ['Eixample', 'Gràcia', 'Poblenou', 'Gòtic', 'Sarrià', 'Sants']
CUISINES = ['Burgers', 'Sushi', 'Pizza', 'Healthy/Salads', 'Mexican', 'Tacos']
VEHICLE_TYPES = ['Bicycle', 'Moto', 'Electric Scooter']

print("Initialization complete. Starting generation...")

Initialization complete. Starting generation...


In [28]:
#Generate restaurant data
restaurant_data = []

for i in range(1,51):
    restaurant_data.append({
        'restaurant_id': f'REST_{i:03d}',
        'restaurant_name':  f'Partner Kitchen{i}',
        'neighborhood': np.random.choice(NEIGHBORHOODS),
        'cuisine_type': np.random.choice(CUISINES),
        'avg_prep_time_baseline': np.random.randint(10, 25)

    })

df_restaurants = pd.DataFrame(restaurant_data)

#Generate couries data
courier_data = []
for i in range(1, 151):  # 150 active couriers
    courier_data.append({
        'courier_id': f'COUR_{i:03d}',
        'vehicle_type': np.random.choice(VEHICLE_TYPES, p=[0.4, 0.4, 0.2]), # 40% Bike, 40% Moto, 20% Scooter
        'courier_rating': round(np.random.uniform(4.2, 5.0), 2)
    })

df_couriers = pd.DataFrame(courier_data)

In [29]:
#df_restaurants.head()
df_couriers.head()

,courier_id,vehicle_type,courier_rating
0,COUR_001,Moto,4.39
1,COUR_002,Bicycle,4.23
2,COUR_003,Moto,4.29
3,COUR_004,Moto,4.36
4,COUR_005,Electric Scooter,4.58


In [30]:
#Generate interconnected orders data 
orders_list = []

for order_id in range(1,NUM_ORDERS + 1):
    #Randomly assign a restaurant and courier for a transaction
    assigned_rest = df_restaurants.sample(1).iloc[0]
    assigned_cour = df_couriers.sample(1).iloc[0]

    #Simulate order placement time over a 7-day period

    random_days = np.random.randint(0,7)
    random_hours = np.random.choice(
    range(24),
    p=[
        0.02, 0.005, 0.005, 0.001, 0.001, 0.008, 0.02, 0.05,
        0.08, 0.05, 0.04, 0.04, 0.06, 0.07, 0.06, 0.04,
        0.04, 0.06, 0.10, 0.11, 0.06, 0.04, 0.03, 0.01
    ]
)#Weights peak heavily around lunch (13-14h) and dinner (19-20h)
    random_minutes = np.random.randint(0,60)

    order_placed = START_DATE + timedelta(days=random_days, hours=int(random_hours), minutes=random_minutes)

    #Weather operational impact (15% chance of rain in this simulation)
    is_raining = np.random.choice([0,1], p=[0.85, 0.15])

    #Calculate Operational Times (Minutes)
    #If raining or peak hours, preparation and travel times increase
    is_peak = 1 if random_hours in [13, 14, 19, 20] else 0 
    prep_multiplier = 1.3 if is_peak else 1.0

    prep_time = round(assigned_rest['avg_prep_time_baseline'] * prep_multiplier + np.random.normal(2, 3))
    prep_time = max(5, prep_time)#Minimum 5 mins to cook

    #Travel time influenced by vehicle type and weather
    base_travel = np.random.randint(8,22)
    weather_delay = np.random.randint(5,12) if (is_raining and assigned_cour['vehicle_type'] == 'Bicycle') else 0
    travel_time = base_travel + weather_delay 

    #Courier wait time at restaurant (negative means courier arrived late, food was waiting)
    courier_wait_at_rest = round(np.random.normal(4,3)) if not is_peak else round(np.random.normal(8,4))
    courier_wait_at_rest = max(-2,courier_wait_at_rest)

    #Financial metrics 
    basket_value = round(np.random.uniform(12.50, 65.00), 2)
    delivery_fee = round(np.random.uniform(1.50, 4.50), 2)
    commission_earned = round(basket_value * 0.22, 2) #Glovo takes roughly 22% commission from restaurants

    #Order Status Logic (Simulate real failures)
    #2% chance order gets canceled by user/restaurant before delivery
    order_status = np.random.choice(['Delivered', 'Cancelled'], p=[0.98, 0.02])

    orders_list.append({
        'order_id': f'GLV_{order_id:06d}',
        'order_placed_at': order_placed,
        'restaurant_id': assigned_rest['restaurant_id'],
        'courier_id': assigned_cour['courier_id'],
        'delivery_neighborhood': np.random.choice(NEIGHBORHOODS),
        'basket_value_euros': basket_value if order_status == 'Delivered' else 0,
        'delivery_fee_euros': delivery_fee if order_status == 'Delivered' else 0,
        'commission_euros': commission_earned if order_status == 'Delivered' else 0,
        'prep_time_minutes': prep_time,
        'courier_wait_minutes': courier_wait_at_rest if order_status == 'Delivered' else 0,
        'travel_time_minutes': travel_time if order_status == 'Delivered' else 0,
        'is_raining': is_raining,
        'order_status': order_status
    })

df_orders = pd.DataFrame(orders_list)

#EXPORT TO CSV FOR SQL INGESTION 
df_restaurants.to_csv('glovo_restaurants.csv', index=False)
df_couriers.to_csv('glovo_couriers.csv', index=False)
df_orders.to_csv('glovo_orders_1.csv', index=False)

print(f"Success! Generated {len(df_orders)} orders and exported 3 clean CSV files.")

Success! Generated 5000 orders and exported 3 clean CSV files.


In [31]:
df_orders.head(20)

,order_id,order_placed_at,restaurant_id,courier_id,delivery_neighborhood,basket_value_euros,delivery_fee_euros,commission_euros,prep_time_minutes,courier_wait_minutes,travel_time_minutes,is_raining,order_status
0,GLV_000001,2026-06-03 10:13:00,REST_018,COUR_004,Gòtic,24.50,3.52,5.39,15,1,11,0,Delivered
1,GLV_000002,2026-06-02 12:18:00,REST_016,COUR_032,Poblenou,44.56,2.37,9.80,15,9,8,0,Delivered
2,GLV_000003,2026-06-05 16:21:00,REST_028,COUR_130,Gòtic,62.96,1.95,13.85,21,0,8,0,Delivered
3,GLV_000004,2026-06-03 20:16:00,REST_009,COUR_016,Eixample,62.80,3.78,13.82,25,7,21,1,Delivered
4,GLV_000005,2026-06-03 20:27:00,REST_047,COUR_021,Sants,49.12,3.54,10.81,31,11,27,1,Delivered
5,GLV_000006,2026-06-07 09:54:00,REST_011,COUR_133,Gràcia,15.23,3.12,3.35,13,7,16,0,Delivered
6,GLV_000007,2026-06-02 17:07:00,REST_024,COUR_051,Sants,48.70,3.53,10.71,18,5,16,1,Delivered
7,GLV_000008,2026-06-07 19:35:00,REST_030,COUR_128,Gòtic,44.90,2.83,9.88,26,5,16,0,Delivered
8,GLV_000009,2026-06-03 14:51:00,REST_011,COUR_056,Sants,37.49,2.63,8.25,14,10,9,0,Delivered
9,GLV_000010,2026-06-07 13:19:00,REST_036,COUR_063,Sants,34.16,3.59,7.52,28,7,13,0,Delivered


In [32]:
#New Orders list, adding courier availability and traffic conditions to the simulation. This will allow for a more realistic predictive model of delivery times, accounting for external factors that can affect courier performance.

In [ ]:
orders_list = []

for order_id in range(1, NUM_ORDERS + 1):
    assigned_rest = df_restaurants.sample(1).iloc[0]
    assigned_cour = df_couriers.sample(1).iloc[0]
    
       #Simulate order placement time over a 7-day period

    random_days = np.random.randint(0,7)
    random_hours = np.random.choice(
    range(24),
    p=[
        0.02, 0.005, 0.005, 0.001, 0.001, 0.008, 0.02, 0.05,
        0.08, 0.05, 0.04, 0.04, 0.06, 0.07, 0.06, 0.04,
        0.04, 0.06, 0.10, 0.11, 0.06, 0.04, 0.03, 0.01
    ]
)#Weights peak heavily around lunch (13-14h) and dinner (19-20h)
    random_minutes = np.random.randint(0, 60)
    order_placed = START_DATE + timedelta(days=random_days, hours=int(random_hours), minutes=random_minutes)
    
    is_raining = np.random.choice([0, 1], p=[0.85, 0.15])
    is_peak = 1 if random_hours in [13, 14, 19, 20] else 0
    
    
    # NEW MARKETPLACE LEVER 1: TRAFFIC GRIDLOCK CONDITIONS
   
    # Central/dense zones have terrible traffic multipliers during peak rush hours
    dense_neighborhoods = ['Eixample', 'Gòtic']
    if is_peak and (assigned_rest['neighborhood'] in dense_neighborhoods):
        traffic_multiplier = 1.6  # 60% slower due to city center traffic jams
    elif is_peak:
        traffic_multiplier = 1.2  # 20% slower in outer neighborhoods
    else:
        traffic_multiplier = 1.0  # Clear roads
        
   
    # NEW MARKETPLACE LEVER 2: COURIER AVAILABILITY (SUPPLY/DEMAND)

    # Simulate systemic vehicle deficits. Bicycles are scarce when it rains.
    # High demand hours (is_peak) across popular districts cause courier wait times to climb.
    if is_peak:
        fleet_scarcity_minutes = np.random.randint(8, 18) # High demand, fewer free riders
    elif is_raining and assigned_cour['vehicle_type'] == 'Bicycle':
        fleet_scarcity_minutes = np.random.randint(10, 20) # Riders log off when it rains
    else:
        fleet_scarcity_minutes = np.random.randint(1, 5) # Abundant couriers nearby
        
    # --- CALCULATING THE CHRONOLOGICAL TIMELINE ---
    # 1. Kitchen Prep Time
    prep_multiplier = 1.3 if is_peak else 1.0
    prep_time = round(assigned_rest['avg_prep_time_baseline'] * prep_multiplier + np.random.normal(1, 2))
    prep_time = max(5, prep_time)
    
    # 2. Courier Wait Time (Driven directly by our new fleet scarcity variable)
    courier_wait_at_rest = fleet_scarcity_minutes + round(np.random.normal(2, 2))
    courier_wait_at_rest = max(0, courier_wait_at_rest)
    
    # 3. Travel Time (Driven directly by our new traffic variable)
    base_travel = np.random.randint(8, 18)
    travel_time = round(base_travel * traffic_multiplier)
    
    # Financials & Status
    basket_value = round(np.random.uniform(12.50, 65.00), 2)
    delivery_fee = round(np.random.uniform(1.50, 4.50), 2)
    commission_earned = round(basket_value * 0.22, 2)
    order_status = np.random.choice(['Delivered', 'Cancelled'], p=[0.98, 0.02])
    
    orders_list.append({
        'order_id': f'GLV_{order_id:06d}',
        'order_placed_at': order_placed,
        'restaurant_id': assigned_rest['restaurant_id'],
        'courier_id': assigned_cour['courier_id'],
        'delivery_neighborhood': np.random.choice(NEIGHBORHOODS),
        'basket_value_euros': basket_value if order_status == 'Delivered' else 0,
        'delivery_fee_euros': delivery_fee if order_status == 'Delivered' else 0,
        'commission_euros': commission_earned if order_status == 'Delivered' else 0,
        'prep_time_minutes': prep_time,
        'courier_wait_minutes': courier_wait_at_rest if order_status == 'Delivered' else 0,
        'travel_time_minutes': travel_time if order_status == 'Delivered' else 0,
        'is_raining': is_raining,
        'order_status': order_status
    })

df_orders = pd.DataFrame(orders_list)
df_orders.to_csv('glovo_orders_2.csv', index=False)
print("New system-driven dataset exported successfully!")

New system-driven dataset exported successfully!


In [ ]:
do i have to change all my sql now? What i finally did, was that i created 2 csv, and left them so the recruiter can see the changes